# 05 — GraphSAGE + LSTM Baseline (MCMIPF on-the-fly)

## Purpose
Train a baseline spatio-temporal model:
- Spatial encoder: GraphSAGE over a PATCH×PATCH graph
- Temporal encoder: LSTM over the history window
- Target: GHI at t_target (6h ahead)

## Import and settings

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path("..").resolve()
DATASET_ROOT = PROJECT_ROOT / "data" / "datasets" / "manifest_v1"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

## Choose sites and load manifests

In [ ]:
SITE = "uniandes"  # change to "elpaso" when needed

SITE_DIR = DATASET_ROOT / SITE
assert SITE_DIR.exists(), f"Missing dataset directory: {SITE_DIR}"

with open(SITE_DIR / "dataset_meta.json", "r", encoding="utf-8") as f:
    meta = json.load(f)

print("Loaded dataset_meta.json:")
print(json.dumps(meta, indent=2))

MCMIPF_ROOT = Path(meta["mcmipf_root"])
FREQ_MIN = int(meta["freq_min"])
H = int(meta["horizon_steps"])
L = int(meta["history_steps"])

GRID_SIZE = int(meta["grid_size"])
SITE_CENTER = (int(meta["site_center_pix"]["row"]), int(meta["site_center_pix"]["col"]))

PATCH = 32
HALF = PATCH // 2

train_man = pd.read_parquet(SITE_DIR / "manifest_train.parquet")
val_man   = pd.read_parquet(SITE_DIR / "manifest_val.parquet")
test_man  = pd.read_parquet(SITE_DIR / "manifest_test.parquet")

print("train:", train_man.shape)
print("val:  ", val_man.shape)
print("test: ", test_man.shape)

## Edge index

In [ ]:
def build_edge_index_8n(patch: int) -> torch.Tensor:
    edges = []
    def node_id(rr: int, cc: int) -> int:
        return rr * patch + cc

    for rr in range(patch):
        for cc in range(patch):
            u = node_id(rr, cc)
            for dr in (-1, 0, 1):
                for dc in (-1, 0, 1):
                    if dr == 0 and dc == 0:
                        continue
                    r2 = rr + dr
                    c2 = cc + dc
                    if 0 <= r2 < patch and 0 <= c2 < patch:
                        v = node_id(r2, c2)
                        edges.append((u, v))

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()  # (2, E)
    return edge_index

EDGE_INDEX = build_edge_index_8n(PATCH)
print("EDGE_INDEX:", EDGE_INDEX.shape)


## MCMPIF 
Loand and extract Patch

In [ ]:
RX = re.compile(r"(?P<ymd>\d{8})_(?P<h>\d{2})_MCMIPF\.npz$")

def hour_path_for_timestamp(t: pd.Timestamp) -> Path:
    ymd = t.strftime("%Y%m%d")
    hh = t.strftime("%H")
    year = t.strftime("%Y")
    month = t.strftime("%m")
    fname = f"{ymd}_{hh}_MCMIPF.npz"
    path = MCMIPF_ROOT / year / month / fname
    return path

def slot_for_timestamp(t: pd.Timestamp) -> int:
    return int(t.strftime("%M")) // 10  # 0..5

def extract_patch(frame_c_hw: np.ndarray, center_rc: tuple[int, int], patch: int) -> np.ndarray:
    r0, c0 = center_rc
    half = patch // 2
    r1, r2 = r0 - half, r0 + half
    c1, c2 = c0 - half, c0 + half

    if (r1 < 0) or (c1 < 0) or (r2 > GRID_SIZE) or (c2 > GRID_SIZE):
        raise ValueError(f"Patch exceeds bounds: rows [{r1},{r2}) cols [{c1},{c2})")

    return frame_c_hw[:, r1:r2, c1:c2]  # (C, P, P)

def patch_to_node_features(patch_c_pp: np.ndarray) -> np.ndarray:
    # (C, P, P) -> (P*P, C)
    C, P1, P2 = patch_c_pp.shape
    x = np.transpose(patch_c_pp, (1, 2, 0)).reshape(P1 * P2, C).astype(np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    return x


## Helpers

In [ ]:
class GraphSeqDataset(Dataset):
    def __init__(self, manifest: pd.DataFrame, site_center: tuple[int,int]):
        self.man = manifest.reset_index(drop=True)
        self.site_center = site_center

    def __len__(self) -> int:
        return len(self.man)

    def __getitem__(self, i: int):
        row = self.man.iloc[i]
        y = float(row["y"])
        history_ts = row["history_ts"]

        # history_ts is stored as list of strings
        if isinstance(history_ts, str):
            # parquet sometimes stores lists as strings depending on engine
            # expect JSON-like string
            history_ts = json.loads(history_ts)

        xs = []
        for ts_str in history_ts:
            # t = pd.Timestamp(ts_str).tz_localize("UTC") if "+" not in ts_str else pd.Timestamp(ts_str)
            t = pd.Timestamp(ts_str)
            if t.tzinfo is None:
                t = t.tz_localize("UTC")
            else:
                t = t.tz_convert("UTC")

            t = t.tz_convert("UTC")

            p = hour_path_for_timestamp(t)
            slot = slot_for_timestamp(t)

            # data = np.load(p)
            # tensor = data["mcmipf"]  # (6, 16, 256, 256)
            # frame = tensor[slot]     # (16, 256, 256)
            
            with np.load(p) as data:
                tensor = data["mcmipf"]
                frame = tensor[slot]


            patch = extract_patch(frame, self.site_center, PATCH)  # (16, 32, 32)
            x = patch_to_node_features(patch)                      # (1024, 16)

            xs.append(x)

        x_seq = np.stack(xs, axis=0)  # (L, N, C)
        return torch.from_numpy(x_seq), torch.tensor(y, dtype=torch.float32)

train_ds = GraphSeqDataset(train_man, SITE_CENTER)
val_ds   = GraphSeqDataset(val_man, SITE_CENTER)
test_ds  = GraphSeqDataset(test_man, SITE_CENTER)

print("train_ds:", len(train_ds), "val_ds:", len(val_ds), "test_ds:", len(test_ds))


## Data Load

In [ ]:
BATCH_SIZE = 2
NUM_WORKERS = 0

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                          pin_memory=(DEVICE=="cuda"))
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                          pin_memory=(DEVICE=="cuda"))
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                          pin_memory=(DEVICE=="cuda"))


## GraphSage
custom-made -> no external libs

In [ ]:
class GraphSAGELayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_nei  = nn.Linear(in_dim, out_dim)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        # x: (N, F), edge_index: (2, E) with edges u->v
        N = x.size(0)
        src, dst = edge_index[0], edge_index[1]  # u, v

        nei_sum = torch.zeros((N, x.size(1)), device=x.device, dtype=x.dtype)
        nei_cnt = torch.zeros((N, 1), device=x.device, dtype=x.dtype)

        nei_sum.index_add_(0, dst, x[src])
        nei_cnt.index_add_(0, dst, torch.ones((dst.numel(), 1), device=x.device, dtype=x.dtype))

        nei_mean = nei_sum / (nei_cnt + 1e-6)

        out = self.lin_self(x) + self.lin_nei(nei_mean)
        return torch.relu(out)


## Model
**GraphSAGE per frame + LSTM over time**

In [ ]:
class GraphSAGE_LSTM(nn.Module):
    def __init__(self, in_dim: int, hidden_g: int, hidden_t: int, out_dim: int = 1):
        super().__init__()
        self.g1 = GraphSAGELayer(in_dim, hidden_g)
        self.g2 = GraphSAGELayer(hidden_g, hidden_g)

        # temporal encoder over pooled graph embeddings
        self.lstm = nn.LSTM(input_size=hidden_g, hidden_size=hidden_t, batch_first=True)

        self.head = nn.Sequential(
            nn.Linear(hidden_t, hidden_t),
            nn.ReLU(),
            nn.Linear(hidden_t, out_dim)
        )

    def forward(self, x_seq: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        # x_seq: (B, L, N, C)
        B, Ls, N, C = x_seq.shape
        edge_index = edge_index.to(x_seq.device)

        # process each time step
        embeds = []
        for t in range(Ls):
            x = x_seq[:, t]  # (B, N, C)

            # apply graph encoder per sample
            # (loop over batch to keep implementation simple and correct)
            pooled = []
            for b in range(B):
                xb = x[b]                  # (N, C)
                h = self.g1(xb, edge_index)
                h = self.g2(h, edge_index)
                g = h.mean(dim=0)          # global mean pooling -> (hidden_g,)
                pooled.append(g)
            pooled = torch.stack(pooled, dim=0)  # (B, hidden_g)
            embeds.append(pooled)

        z = torch.stack(embeds, dim=1)  # (B, L, hidden_g)
        out, _ = self.lstm(z)           # (B, L, hidden_t)
        last = out[:, -1]               # (B, hidden_t)
        yhat = self.head(last).squeeze(-1)  # (B,)
        return yhat


## Training

In [ ]:
def rmse_torch(yhat: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    return torch.sqrt(torch.mean((yhat - y) ** 2))

model = GraphSAGE_LSTM(in_dim=16, hidden_g=64, hidden_t=64).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

EPOCHS = 3  # first sanity run

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_losses = []

    for x_seq, y in train_loader:
        x_seq = x_seq.to(DEVICE)   # (B,L,N,C)
        y = y.to(DEVICE)

        opt.zero_grad()
        yhat = model(x_seq, EDGE_INDEX)
        loss = loss_fn(yhat, y)
        loss.backward()
        opt.step()

        tr_losses.append(loss.item())

    model.eval()
    va_losses = []
    va_rmses = []

    with torch.no_grad():
        for x_seq, y in val_loader:
            x_seq = x_seq.to(DEVICE)
            y = y.to(DEVICE)
            yhat = model(x_seq, EDGE_INDEX)
            va_losses.append(loss_fn(yhat, y).item())
            va_rmses.append(rmse_torch(yhat, y).item())

    print(f"Epoch {epoch:02d} | train_mse={np.mean(tr_losses):.5f} | val_mse={np.mean(va_losses):.5f} | val_rmse={np.mean(va_rmses):.5f}")
